In [2]:
import xml.etree.ElementTree as ET
import unicodedata
from pathlib import Path
import os
import pandas as pd

ET.VERSION

'1.3.0'

### paraNLP abstracts

In [25]:
# adapted from text2tmx_doc in format_parallel2tmx_LREC2026.ipynb
def text2tmx_doc(src, tgt, docid, src_aligned_path, tgt_aligned_path, store_path, name = 'rTAL', date = '2023-04-28'):
    # src = 'FR', tgt = 'EN', 
    # assume that src file and tgt file have a format s. t. <sep_tok> \n <docid> \n parallel sents

    # the src and target sentences
    with open(src_aligned_path, 'r', encoding='utf-8') as f:
        src_sents = f.read().split('\n') 

    with open(tgt_aligned_path, 'r', encoding='utf-8') as f:
        tgt_sents = f.read().split('\n') 

    assert len(src_sents) == len(tgt_sents), f"Mismatch: {len(src_sents)} src vs {len(tgt_sents)} tgt in {docid}"

    sentences_src = []
    sentences_tgt = []
    for s, t in zip(src_sents, tgt_sents):
        if s.strip() and t.strip():
            sentences_src.append(s)
            sentences_tgt.append(t)
    assert len(sentences_src) == len(sentences_tgt)


    # Create the root element of the TMX document
    tmx = ET.Element('tmx', version='1.4b')

    header_attrib = { 
        'creationtool' : 'xml.etree.ElementTree',
        'creationtoolversion' : '1.3.0',
        'datatype': "PlainText",
        'segtype' : "sentence",
        'adminlang': "en-us",
        'srclang': src,
        'o-tmf': "XML",
        'creationdate': date,
        'creationid': "MaTOS"
    }
    
    header = ET.SubElement(tmx, 'header', header_attrib)

    # Set the language codes for the source and target languages
    etm = ET.SubElement(header, 'note')
    etm.text = f"This is the sentence alignment file for {name}-{docid}." + " segId begin by 1, tuid = segId"
    etm = ET.SubElement(header, 'docid')
    etm.text = docid
    
    etm = ET.SubElement(header, 'elem', type='sourceLanguage')
    etm.text = src
    etm = ET.SubElement(header, 'elem', type='targetLanguage')
    etm.text = tgt

    # Create a new body element for the translation units
    body = ET.SubElement(tmx, 'body')
    
    seg_id = 1
    XML_LANG = '{http://www.w3.org/XML/1998/namespace}lang'
    # Iterate through the sentence pairs and add them to the TMX document
    for s, t in zip(sentences_src, sentences_tgt):    
        
        tu = ET.SubElement(body, 'tu', {'tuid': str(seg_id)})
        tuv_src = ET.SubElement(tu, 'tuv', {XML_LANG:src})
        seg_src = ET.SubElement(tuv_src, 'seg')
        seg_src.text = unicodedata.normalize('NFC', s )

        tuv_tgt = ET.SubElement(tu, 'tuv', {XML_LANG:tgt})
        seg_tgt = ET.SubElement(tuv_tgt, 'seg')
        seg_tgt.text = unicodedata.normalize('NFC', t )
        
        seg_id +=1

    # Write the TMX document to a file
    tree = ET.ElementTree(tmx)
    ET.indent(tree, space="    ", level=0)
    with open(store_path, 'wb') as f : 
        tree.write(f, encoding='utf-8', xml_declaration=True)



def text2tmx_doc_all(src, tgt, align_dir, aligned_fname_prefix, store_dir, store_fname, ignore_list = [], name = 'rTAL', date = '2023-04-28'):
    # doc_path refers to a doc with several folders 

    for doc in os.listdir(align_dir):
        if doc in ignore_list:
            continue

        doc_dir = os.path.join(align_dir, doc)
        src_aligned_path = os.path.join( doc_dir, f"{aligned_fname_prefix}.{src.lower()}.txt")
        tgt_aligned_path = os.path.join( doc_dir, f"{aligned_fname_prefix}.{tgt.lower()}.txt")
        
        if os.path.exists(src_aligned_path) and os.path.exists(tgt_aligned_path)  :
            store_path = os.path.join(store_dir, doc, store_fname)
            Path(os.path.dirname(store_path)).mkdir(parents=True, exist_ok=True)
            
            text2tmx_doc(
                src,
                tgt, 
                docid = doc, 
                src_aligned_path = src_aligned_path, 
                tgt_aligned_path = tgt_aligned_path,
                store_path = store_path,
                name = name,
                date = date,
            )
        else:
            
            if os.path.isdir(doc_dir) and doc[0] != '.':
                print(f"Not found src or tgt alignments for {doc}")

In [26]:
src = 'FR'
tgt = 'EN'

name = 'ISTEX'
align_dir = 'TAL/data_LREC2026/abstract_m_trankit'

store_dir = f'TAL/data_LREC2026/tmx/{name}'
to_rm_path = f'{align_dir}/drop.list.txt'
TO_REMOVE = [ l.strip() for l in open(to_rm_path, 'r').read().strip().split('\n') if l[0] != '#']
print(TO_REMOVE)

text2tmx_doc_all(
    src, 
    tgt, 
    align_dir = align_dir, 
    aligned_fname_prefix =f'bertalign_abstract_m', 
    store_dir = store_dir,
    store_fname = f'MaTOS_{name}_aligned.tmx', 
    ignore_list = TO_REMOVE, 
    name = name,
    date = '2026-03-05',
)




['5DACF7B626EC8AAFBE291D6750BF1E2F1C1B35D4', 'E57415C9B50071A4B6183E46C97E7E5F9DA19727', '4D9972E5E7D9E06FC90FC8FA0B7F6033FED7B194', 'B29FAF9A9068734D20161668DC8D570DD7BA9C50', '1D6CA49C4B5727E38B044743F7CD0331ED057E1E', '0C4EA933A27173347BD71A04596B9D69F035A385', '0E4E8AD8D037D8FCB0C79C48FCEE70A2F427A2AB', '568C0529E242652CAE3A1F90D8FDEF3EDF867B02', 'E964B87D9D803323AD7564AFECA93E323D2DCDA3', '6244B7A12574380E772BEAE5015111864ACDBB65', '7406DF8EF4AAA19E3658587A5AD5F610026BD647']


In [27]:
src = 'FR'
tgt = 'EN'

name = 'THE'
align_dir = 'TAL/data_LREC2026/theses.fr'


store_dir = f'TAL/data_LREC2026/tmx/{name}'
to_rm_path = f'{align_dir}/drop.list.txt'
TO_REMOVE = [ l.strip() for l in open(to_rm_path, 'r').read().strip().split('\n') if l[0] != '#']
print(TO_REMOVE)


text2tmx_doc_all(
    src, 
    tgt, 
    align_dir = align_dir, 
    aligned_fname_prefix =f'bertalign_these', 
    store_dir = store_dir,
    store_fname = f'MaTOS_{name}_aligned.tmx', 
    ignore_list = TO_REMOVE, 
    name = name,
    date = '2026-03-05',
)


['2004PA132039', '2017USPCD049', 's256511', '2012GRENM010', '2018REN1S041', 's251443', '2018MONTS024', 's199212']


### NLPsilver

In [34]:

def text2tmx_articles(src, tgt, docid, meta_dict, src_aligned_path, tgt_aligned_path, store_path, name = 'NLPsilver', date = '2026-03-28'):
    # assume that src file and tgt file have a format s. t. <sep_tok> \n <docid> \n parallel sents

    # the src and target sentences
    with open(src_aligned_path, 'r', encoding='utf-8') as f:
        sentences_src = [ l.strip() for l in f.read().strip().split('\n') if l.strip()]

    with open(tgt_aligned_path, 'r', encoding='utf-8') as f:
        sentences_tgt = [ l.strip() for l in f.read().strip().split('\n') if l.strip()]
        
    assert(len(sentences_src) == len(sentences_tgt))


    # Create the root element of the TMX document
    tmx = ET.Element('tmx', version='1.4b')

    header_attrib = { 
        'creationtool' : 'xml.etree.ElementTree',
        'creationtoolversion' : '1.3.0',
        'datatype': "PlainText",
        'segtype' : "sentence",
        'adminlang': "en-us",
        'srclang': src,
        'o-tmf': "XML",
        'creationdate': date,
        'creationid': "MaTOS"
    }


    header = ET.SubElement(tmx, 'header', header_attrib)

    XML_LANG = '{http://www.w3.org/XML/1998/namespace}lang'

    # Set the language codes for the source and target languages
    etm = ET.SubElement(header, 'note')
    etm.text = f"This is the sentence alignement file for {name}-{docid}." + " segId begin by 1, tuid = segId"
    for k in ['url_en', 'url_fr', 'title_en', 'title_fr']:
        etm = ET.SubElement(header, k)
        etm.text = meta_dict[k] 
    etm = ET.SubElement(header, 'docid')
    etm.text = docid
    
    etm = ET.SubElement(header, 'elem', type='sourceLanguage')
    etm.text = src
    etm = ET.SubElement(header, 'elem', type='targetLanguage')
    etm.text = tgt

    # Create a new body element for the translation units
    body = ET.SubElement(tmx, 'body')
    
    seg_id = 1
    # Iterate through the sentence pairs and add them to the TMX document
    for s, t in zip(sentences_src, sentences_tgt):    
        
        tu = ET.SubElement(body, 'tu', {'tuid': str(seg_id)})
        tuv_src = ET.SubElement(tu, 'tuv', {XML_LANG:src})
        seg_src = ET.SubElement(tuv_src, 'seg')
        seg_src.text = unicodedata.normalize('NFC', s )

        tuv_tgt = ET.SubElement(tu, 'tuv', {XML_LANG:tgt})
        seg_tgt = ET.SubElement(tuv_tgt, 'seg')
        seg_tgt.text = unicodedata.normalize('NFC', t )
        
        seg_id +=1

    # Write the TMX document to a file
    tree = ET.ElementTree(tmx)
    ET.indent(tree, space="    ", level=0)
    with open(store_path, 'wb') as f : 
        tree.write(f, encoding='utf-8', xml_declaration=True)




def text2tmx_article_all(src, tgt, meta_df, align_dir, aligned_fname_prefix, store_dir, store_fname, ignore_list = [], name = 'NLPsilver', date = '2026-03-28'):
    # doc_path refers to a doc with several folders 

    for doc in os.listdir(align_dir):
        if doc in ignore_list:
            continue

        doc_dir = os.path.join(align_dir, doc)
        src_aligned_path = os.path.join( doc_dir, f"{aligned_fname_prefix}.{src.lower()}.txt")
        tgt_aligned_path = os.path.join( doc_dir, f"{aligned_fname_prefix}.{tgt.lower()}.txt")
        
        if os.path.exists(src_aligned_path) and os.path.exists(tgt_aligned_path)  :
            # print(doc)
            store_path = os.path.join(store_dir, doc, store_fname)
            Path(os.path.dirname(store_path)).mkdir(parents=True, exist_ok=True)

            meta_dict = meta_df.loc[doc].to_dict()
            if doc == 'taln-2009-court-039':
                meta_dict['url_en'] = 'None'
            text2tmx_articles(
                src,
                tgt, 
                docid = doc, 
                meta_dict = meta_dict ,
                src_aligned_path = src_aligned_path, 
                tgt_aligned_path = tgt_aligned_path,
                store_path = store_path,
                name = name,
                date = date,
            )
            
            stat_path = os.path.join( doc_dir, f"{aligned_fname_prefix}.{tgt.lower()}.txt.stat.tsv")
            if doc == 'taln-2009-court-039':
                stat_path = os.path.join( doc_dir, f"{aligned_fname_prefix}.{src.lower()}-{tgt.lower()}.stat.tsv")
            
            stat_to_store_path = os.path.join(store_dir, doc, f"{name}_{src.lower()}_{tgt.lower()}.{tgt.lower()}.stat.tsv")
            os.system(f"cp {stat_path} {stat_to_store_path}")
                
        else:
            
            if os.path.isdir(doc_dir) and doc[0] != '.':
                print(f"Not found src or tgt alignments for {doc}")

In [16]:

meta_df = pd.read_csv("TAL/corpora/ACL_update202603/ACL_comparable_corpus_resources.tsv", sep = '\t', index_col = 0)
meta_df.loc['taln-2009-court-039'].to_dict()

{'docid-en': nan,
 'docid': 'taln-2009-court-039',
 'title_en': 'Automatic Processing of Disfluencies in Linguistically Constrained Spoken Dialogs Corpus',
 'url_en': nan,
 'title_fr': 'Traitement automatique de disfluences dans un corpus linguistiquement contraint de dialogue oral spontané',
 'url_fr': 'https://aclanthology.org/2009.jeptalnrecital-court.39.pdf',
 'misc': nan}

In [35]:
src = 'EN'
tgt = 'FR'

name = 'NLPsilver'
sub_dir = 'TAL/corpora/ACL_update202603/ACL_v3_2026-03'
align_dir = f'{sub_dir}/ACL_corpus/{src.lower()}_{tgt.lower()}'

store_dir =f'{sub_dir}/ACL_corpus_tmx/{src.lower()}_{tgt.lower()}'


TO_REMOVE = [ ] 
print(TO_REMOVE)



text2tmx_article_all(
    src, 
    tgt,
    meta_df,
    align_dir = align_dir, 
    aligned_fname_prefix =f'acl_corpus', 
    store_dir = store_dir,
    store_fname = f'MaTOS_{name}_aligned.tmx', 
    ignore_list = TO_REMOVE, 
    name = name,
    date = '2026-03-30',
)
 



[]
Not found src or tgt alignments for F04-25.revised
Not found src or tgt alignments for F04-46.revised
Not found src or tgt alignments for F08-1586.revised


In [36]:
src = 'FR'
tgt = 'EN'

name = 'NLPsilver'
sub_dir = 'TAL/corpora/ACL_update202603/ACL_v3_2026-03'
align_dir = f'{sub_dir}/ACL_corpus/{src.lower()}_{tgt.lower()}'

store_dir =f'{sub_dir}/ACL_corpus_tmx/{src.lower()}_{tgt.lower()}'


TO_REMOVE = [ ] 
print(TO_REMOVE)

text2tmx_article_all(
    src, 
    tgt,
    meta_df,
    align_dir = align_dir, 
    aligned_fname_prefix =f'acl_corpus', 
    store_dir = store_dir,
    store_fname = f'MaTOS_{name}_aligned.tmx', 
    ignore_list = TO_REMOVE, 
    name = name,
    date = '2026-03-30',
)
 


[]


## paraEPS abstracts

In [2]:
import xml.etree.ElementTree as ET
import unicodedata
from pathlib import Path
import os
import pandas as pd
ET.VERSION

# copied and adapted from the scripts of Lichao
def read_source_doc_df(doc_path): 
    if doc_path.endswith('tsv') : 
        df = pd.read_csv(doc_path, sep="\t",  encoding="utf-8")     
        return df


def text2tmx_doc_step(src, tgt, docid, meta_dict, src_aligned_path, tgt_aligned_path, store_path, name = 'CRGEOS',  date = '2026-03-28'):
    # Create the root element of the TMX document
    tmx = ET.Element('tmx', version='1.4b')
    header_attrib = { 
        'creationtool' : 'xml.etree.ElementTree',
        'creationtoolversion' : '1.3.0',
        'datatype': "PlainText",
        'segtype' : "sentence",
        'adminlang': "en",
        'srclang': src,
        'tgtlang': tgt,
        'o-tmf': "XML",
        'creationdate': date ,
        'creationid': "MaTOS"
    }


    header = ET.SubElement(tmx, 'header', header_attrib)
    # Set the language codes for the source and target languages
    etm = ET.SubElement(header, 'note')
    etm.text = f"This is the sentence alignement file for {name}_{docid}" + " segId begin by 1, tuid = segId"

    etm = ET.SubElement(header, 'docid')
    etm.text = docid
    for m in ["author", "title" , "title_en",  "title_fr" , "citation" , "volume" , "institution", "journal" ]:
        
        if meta_dict.get(m, '') : 
            etm = ET.SubElement(header, 'prop', type=m)
            etm.text = meta_dict[m]
    

    etm = ET.SubElement(header, 'elem', type='sourceLanguage')
    etm.text = src

    etm = ET.SubElement(header, 'elem', type='targetLanguage')
    etm.text = tgt

    #####################################################
    # Create a new body element for the translation units
    body = ET.SubElement(tmx, 'body')
    
    # the src and tgt sentences
    with open(src_aligned_path, 'r', encoding='utf-8') as f:
        src_sents = f.read().split('\n') 

    with open(tgt_aligned_path, 'r', encoding='utf-8') as f:
        tgt_sents = f.read().split('\n') 

    assert len(src_sents) == len(tgt_sents), f"Mismatch: {len(src_sents)} src vs {len(tgt_sents)} tgt in {docid}"
    # print(docid)
    sentences_src = []
    sentences_tgt = []
    for s, t in zip(src_sents, tgt_sents):
        if s.strip() and t.strip():
            sentences_src.append(s)
            sentences_tgt.append(t)
    assert len(sentences_src) == len(sentences_tgt)
    
    
    # Iterate through the sentence pairs and add them to the TMX document
    seg_id = 1
    XML_LANG = '{http://www.w3.org/XML/1998/namespace}lang'
    for s, t in zip(sentences_src, sentences_tgt):    
        
        tu = ET.SubElement(body, 'tu', {'tuid': str(seg_id)})
        tuv_src = ET.SubElement(tu, 'tuv', {XML_LANG:src})
        seg_src = ET.SubElement(tuv_src, 'seg')
        seg_src.text = s
        
        tuv_tgt = ET.SubElement(tu, 'tuv', {XML_LANG:tgt})
        seg_tgt = ET.SubElement(tuv_tgt, 'seg')
        seg_tgt.text = t
        
        seg_id +=1

    # Write the TMX document to a file
    tree = ET.ElementTree(tmx)    
    ET.indent(tree, space="    ", level=0)
    
    with open(store_path, 'wb') as f : 
        tree.write(f, encoding='utf-8', xml_declaration=True)
    
    return tree


def text2tmx_doc_all_step(
        src, 
        tgt, 
        meta_df, 
        align_dir, 
        store_dir, 
        store_fname, 
        name = 'CRAS', 
        date = '2026-03-28'
        ):
    """align_dir contains the aligned EN and FR sentences stored in separate files, with filename {name}_docid.{lang}  """

    if name in [ 'STEP_THE' , 'THE']:
        tmp_nb = len('STEP_THE_')
        fname_list = set([l[tmp_nb:-3] for l in os.listdir(align_dir) if l[0] != '.' ])
        
    else:
        fname_list = set([l.split('_')[1][:-3] for l in os.listdir(align_dir) if l[0] != '.' ])

    log_info = []
    for docid in fname_list :
        if not docid in meta_df.index:
            print(f'{docid} not in meta data of {name}')
            log_info.append(docid)
            continue

        src_aligned_path = os.path.join( align_dir, f"{name}_{docid}.{src.lower()}")
        tgt_aligned_path = os.path.join( align_dir, f"{name}_{docid}.{tgt.lower()}")
        
        if os.path.exists(src_aligned_path) and os.path.exists(tgt_aligned_path)  :

            store_path = os.path.join(store_dir, docid, store_fname)
            Path(os.path.dirname(store_path)).mkdir(parents=True, exist_ok=True)

            meta_dict = meta_df.loc[docid].to_dict()
            
            text2tmx_doc_step(
                src, 
                tgt, 
                docid= docid, 
                meta_dict = meta_dict,
                src_aligned_path = src_aligned_path, 
                tgt_aligned_path = tgt_aligned_path,
                store_path = store_path,
                name = 'THE' if name == 'STEP_THE' else name,
                date = date,
            )

        else:
            print(f"Not found src or tgt alignments for {docid}")
            print(src_aligned_path)
            print(tgt_aligned_path)
            
            log_info.append(docid)
    return log_info

In [43]:




src = 'EN'
tgt = 'FR'

# for name in [ 'BSGF', 'CRAS', 'CRG', 'STEP_THE', 'ISTEX']:
for name in [ 'STEP_THE']:
    
    
    
    tmp_name = 'THE' if name == 'STEP_THE' else name
    doc_path = f"/Users/ziqianpeng/MaTOS/Resources/MaTOS-corpus-BUCC2026/paraEPS/tsv_raw/{tmp_name}/{tmp_name}.tsv"
    meta_df = read_source_doc_df(doc_path).set_index('docid').fillna('')
    
    
    align_dir = f'STEP/corpora/data_aligned_STEP_LREC/{name}'
    store_dir =f'STEP/corpora/data_aligned_STEP_LREC_xml/{tmp_name}'
    
    
    to_check_docid = text2tmx_doc_all_step(
            src, 
            tgt, 
            meta_df, 
            align_dir = align_dir,  
            store_dir = store_dir,
            store_fname = f'MaTOS_{tmp_name}_aligned.tmx',  
            name = name, 
            date = '2026-03-05'
            )

    print(name)
    print(to_check_docid)


STEP_THE
[]


In [18]:
# name = 'CanMin'
name = 'CJES'

doc_path = f"/Users/ziqianpeng/MaTOS/Resources/MaTOS-corpus-BUCC2026/paraEPS/tsv_raw/{name}/{name}.tsv"
meta_df = read_source_doc_df(doc_path)#.drop(columns = ['abstract_en', 'abstract_fr'] )
# meta_df.to_csv(doc_path, sep = '\t')

meta_df
# meta_df = pd.read_csv(doc_path, sep = '\t', index_col = 0)
# meta_df.loc['bsgf170015.html'].to_dict()

,Unnamed: 0,docid,citation
0,0,E07-043.html,RenbinZhanR. Zhan and JisuoJinJ. Jin. 2008. On...
1,1,E09-068.html,"JohnShawJ. Shaw, DavisSharpeD. Sharpe, and Jef..."
2,2,e84-035.html,M. K. Prabhu and G. R. Webber. 1984. Origin of...
3,3,cjes-2014-0209.html,"Junqing Chen, Xiongqi Pang, and Zhenxue Jiang...."
4,4,e06-029.html,E E Maxwell and M W Caldwell. 2006. Evidence f...
...,...,...,...
4661,4661,e81-066.html,"R. I. Thorpe, Jayanta Guha, and Jules Cimon. 1..."
4662,4662,e00-093.html,"Nancy C Grainger, Michael E Villeneuve, Larry ..."
4663,4663,e94-083.html,"Giselle K. Jakobs, Paul L. Smith, and Howard W..."
4664,4664,e92-183.html,Maatthew I. Leybourne and Nancy A. Van Wagoner...


## MERSENNE

In [24]:
import xml.etree.ElementTree as ET
import unicodedata
from pathlib import Path
import os
import pandas as pd
import re
ET.VERSION


def text2tmx_mersenne(meta_df, idx, aligned_fname, store_path, title_en, title_fr, name = 'MERSENNE', date = '2024-11-15'):
    # src = 'FR', tgt = 'EN', 
    # aligned sentences are in {aligned_fname}.{src} and {aligned_fname}.{tgt}
    src = meta_df.loc[idx]['LO'].upper()
    tgt = 'EN' if src == 'FR' else 'FR'

    docid = meta_df.loc[idx]['docid']
    # the src and target sentences
    with open(f"{aligned_fname}.{src.lower()}", 'rt', encoding='utf-8') as f:
        doc_sep_src = [ l for l in f.read().strip().split('\n') if l.strip()]
    
    with open(f"{aligned_fname}.{tgt.lower()}", 'rt', encoding='utf-8') as f:
        doc_sep_tgt = [ l for l in f.read().strip().split('\n') if l.strip()]
        
    assert(len(doc_sep_src) == len(doc_sep_tgt))

    # Create the root element of the TMX document
    tmx = ET.Element('tmx', version='1.4b')

    header_attrib = { 
        'creationtool' : 'xml.etree.ElementTree',
        'creationtoolversion' : '1.3.0',
        'datatype': "PlainText",
        'segtype' : "sentence",
        'adminlang': "en",
        'srclang': src, 
        'tgtlang': tgt,
        'o-tmf': "XML",
        'creationdate': date,
        'creationid': "MaTOS"
    }

    header = ET.SubElement(tmx, 'header', header_attrib)

    # Set the language codes for the source and target languages
    etm = ET.SubElement(header, 'note')
    etm.text = f"This is the sentence alignement file for {name}-{docid}." + " segId begin by 1, tuid = segId" 
    etm = ET.SubElement(header, 'docid')
    etm.text = docid
    # url_src
    etm = ET.SubElement(header, 'url_src')
    etm.text = meta_df.loc[idx]['Version originale']
    # url_src
    etm = ET.SubElement(header, 'url_tgt')
    etm.text = meta_df.loc[idx]['Traduction']
    # author
    etm = ET.SubElement(header, 'author')
    etm.text = unicodedata.normalize('NFC', re.sub('\xa0', '', meta_df.loc[idx]['Auteur']))
    # title_en
    etm = ET.SubElement(header, 'title_en')
    etm.text = title_en  
    # title_fr
    etm = ET.SubElement(header, 'title_fr')
    etm.text = title_fr
    # volume
    etm = ET.SubElement(header, 'volume')
    # meta_df.loc[0][['Série', 'Année', 'Vol.', 'Thém.']].values
    etm.text =  f"{meta_df.loc[idx]['Série']}, {meta_df.loc[idx]['Année']}, Volume {meta_df.loc[idx]['Vol.']}, {meta_df.loc[idx]['Thém.']}"
    # translator
    etm = ET.SubElement(header, 'translator')
    etm.text = unicodedata.normalize('NFC', re.sub('\xa0', '', meta_df.loc[idx]['Traducteur.trice']))
    # language code
    etm = ET.SubElement(header, 'elem', type='sourceLanguage')
    etm.text = src
    etm = ET.SubElement(header, 'elem', type='targetLanguage')
    etm.text = tgt

    # Create a new body element for the translation units
    body = ET.SubElement(tmx, 'body')
    
    seg_id = 1
    XML_LANG = '{http://www.w3.org/XML/1998/namespace}lang'
    # Iterate through the sentence pairs and add them to the TMX document

    for p_id, src_block in enumerate(doc_sep_src, start=1):
        sents_src = [l.strip() for l in src_block.strip().split('<sep>') if l.strip()]
        sents_tgt = [l.strip() for l in doc_sep_tgt[p_id-1].strip().split('<sep>') if l.strip()]
        assert len(sents_src) == len(sents_tgt)
        for s, t in zip(sents_src, sents_tgt):    
            # id with paragraph boundaries
            tu = ET.SubElement(body, 'tu', {'tuid':str(seg_id)})
            prop = ET.SubElement(tu, "prop", type="paragraph")
            prop.text = str(p_id)

            # sentences
            tuv_src = ET.SubElement(tu, 'tuv', {XML_LANG : src})
            seg_src = ET.SubElement(tuv_src, 'seg')
            seg_src.text = unicodedata.normalize('NFC', s )
    
            tuv_tgt = ET.SubElement(tu, 'tuv', {XML_LANG :tgt})
            seg_tgt = ET.SubElement(tuv_tgt, 'seg')
            seg_tgt.text = unicodedata.normalize('NFC', t )
            
            seg_id +=1


    # Write the TMX document to a file
    tree = ET.ElementTree(tmx)
    ET.indent(tree, space="    ", level=0)
    with open(store_path, 'wb') as f : 
        tree.write(f, encoding='utf-8', xml_declaration=True)
        



In [30]:

# tmp_df = meta_df.reset_index()[['index']].apply(lambda x: f"mersenne{x.values[0]}", axis = 1).to_frame().rename(columns = {0: 'docid'})
# tmp_df
# pd.concat([tmp_df, meta_df], axis = 1).to_csv('STEP/corpora/raw/Mersenne/Traductions-Octobre-2024-geos-chim.csv', sep = '\t')

meta_path = 'STEP/corpora/raw/Mersenne/Traductions-Octobre-2024-geos-chim.csv'
meta_df = pd.read_csv(meta_path, sep = '\t', index_col = 0 )[:23].drop(columns = 'Nbre de mots')

meta_df.to_csv('/Users/ziqianpeng/MaTOS/Resources/MERSENNE-articles-WMT2025/resource_october_2024.tsv', sep = '\t')

meta_df

,docid,Titre,Auteur,Série,Année,Vol.,Thém.,LO,Traducteur.trice,Version originale,Traduction
0,mersenne0,La science n’est pas floue,"Etienne Ghys , Ghislain de Marsily",CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
1,mersenne1,Réchauffement climatique : état des connaissan...,Valérie Masson-Delmotte,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
2,mersenne2,"Le changement climatique, une chance pour l’hu...",Mireille Delmas-Marty,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
3,mersenne3,Changement climatique et éducation,"Pierre Léna , David Wilgenbus",CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
4,mersenne4,« Science des données » versus science physiqu...,Venkatramani Balaji,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
5,mersenne5,Quelles transformations pour l’atténuation du ...,Céline Guivarch,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
6,mersenne6,Le rôle de l’énergie nucléaire dans la lutte c...,Yves Bréchet,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
7,mersenne7,Anticiper l’évolution des territoires,Hervé Le Treut,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
8,mersenne8,Énergie et climat : défis et innovations,Didier Roux,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
9,mersenne9,Changement climatique et biosphère,Isabelle Chuine,CRGEOS,2020,352,"Facing climate change, the range of possibilities",FR,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...


In [26]:
date = '2024-11-15'
# meta_path = 'STEP/corpora/raw/Mersenne/Traductions-Octobre-2024.csv'
# meta_df = pd.read_csv(meta_path, sep = ';' )

DATA_DIR = 'STEP/corpora/raw/Mersenne'
meta_path = 'STEP/corpora/raw/Mersenne/Traductions-Octobre-2024-geos-chim.csv'
meta_df = pd.read_csv(meta_path, sep = '\t', index_col = 0 )[:23]


TXT_DIR = f"{DATA_DIR}/txt/parallel_corpora/Mersenne-groupby-article-renumerate"
ALIGN_DIR = f"{DATA_DIR}/aligned_mersenne_post_processed"

STORE_DIR = f"{DATA_DIR}/tmx/Mersenne-groupby-article-final"
Path(STORE_DIR).mkdir(parents=True, exist_ok=True)


print(len(meta_df))
for idx in range(len(meta_df)):
    docid = meta_df.loc[idx]['docid']
    
    tmp_n = 0
    if idx > 10:
        tmp_n = 1
    if idx > 14:
        tmp_n = 2
    print(idx, idx+tmp_n)
    
    
    title_en = open(f"{ALIGN_DIR}/mersenne{idx+tmp_n}/aligned_title.en", 'r').read().strip()
    title_fr = open(f"{ALIGN_DIR}/mersenne{idx+tmp_n}/aligned_title.fr", 'r').read().strip()

    store_dir = f"{STORE_DIR}/{docid}"
    Path(store_dir).mkdir(parents=True, exist_ok=True)


    aligned_fname = f"{TXT_DIR}/{docid}/doc_auxiliaire/block_sep"
    store_path = f"{store_dir}/MERSENNE_aligned.tmx"
        
    text2tmx_mersenne(
        meta_df, 
        idx, 
        aligned_fname, 
        store_path, 
        title_en, 
        title_fr, 
        name = 'MERSENNE', 
        date = date
    )
        
    try:
        tree = ET.parse(store_path, ET.XMLParser(encoding='utf-8'))
    except:
        print(f"Cannot parse, potentially no text in {fpath}")
        warnings.warn(f"Cannot parse file {fpath} as tree")


23
0 0
1 1
2 2
3 3
4 4
5 5
6 6
7 7
8 8
9 9
10 10
11 12
12 13
13 14
14 15
15 17
16 18
17 19
18 20
19 21
20 22
21 23
22 24


### MERSENNE-GEOS

In [35]:

tmp_dir = "/Users/ziqianpeng/MaTOS/Resources/MaTOS-corpus-BUCC2026"

direction = 'EN-FR'
# meta_path = f"{tmp_dir}/paraEPS/xml_collections/test-long/MERSENNE/Traductions-Octobre-2024-MERSENNE-CRGEOS-{direction}.csv"
# meta_df = pd.read_csv(meta_path, sep = ';').rename(columns = {"Unnamed: 0" : 'index'}).drop(columns = 'Nbre de mots') #, index_col = 0 )
# meta_df

In [32]:
tmp_df = meta_df[['index']].apply(lambda x: f"mersenne{x.values[0]}", axis = 1).to_frame().rename(columns = {0: 'docid'})

store_path = f"{tmp_dir}/paraEPS/xml_collections/test-long/MERSENNE/Traductions-Octobre-2024-CRGEOS-{direction}.csv"
pd.concat([tmp_df, meta_df], axis = 1).set_index('index')#.to_csv(store_path, sep = '\t')


In [36]:
tmp_dir = "/Users/ziqianpeng/MaTOS/Resources/MaTOS-corpus-BUCC2026"
store_path = f"{tmp_dir}/paraEPS/xml_collections/test-long/MERSENNE/Traductions-Octobre-2024-CRGEOS-{direction}.csv"

meta_df = pd.read_csv(store_path, sep = '\t', index_col = ['index'] )
meta_df

,docid,Titre,Auteur,Série,Année,Vol.,Thém.,LO,Traducteur.trice,Version originale,Traduction
index,,,,,,,,,,,
18,mersenne18,Glass as a biomaterial strategies for optimisi...,"Delia Brauer, Leena Hupa",CRGEOS,2022,352,"Le verre, un matériau omniprésent",EN,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
19,mersenne19,"Glass, an ubiquitous material",Daniel R. Neuville,CRGEOS,2022,352,"Le verre, un matériau omniprésent",EN,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
20,mersenne20,Structure from glass to melt: a case study alo...,"Laurent Cormier , Louis Hennet , Gerald Lelong...",CRGEOS,2022,352,"Le verre, un matériau omniprésent",EN,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
21,mersenne21,Distributed fiber optics strain sensors: from ...,"Wilfried Blanc , Luca Schenato , Carlo Molardi...",CRGEOS,2022,352,"Le verre, un matériau omniprésent",EN,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
22,mersenne22,Rheological changes in melts and magmas induce...,"Francesco Vetere , Gianluca Iezzi , Diego Peru...",CRGEOS,2022,352,"Le verre, un matériau omniprésent",EN,Romain Dziegielinski,https://comptes-rendus.academie-sciences.fr/ge...,https://comptes-rendus.academie-sciences.fr/ge...
